# TipSpot

A {class}`~pylabrobot.resources.tip_rack.TipSpot` represents one position in a
{class}`~pylabrobot.resources.TipRack`. The spot is a resource with its own location and
dimensions, while the disposable {class}`~pylabrobot.resources.tip.Tip` that it may contain is a
separate object. Each spot owns a {class}`~pylabrobot.resources.tip_tracker.TipTracker` that records
whether a tip is present.

## Accessing tip spots

Tip spots are normally obtained from a tip rack rather than created directly. Use
{meth}`~pylabrobot.resources.ItemizedResource.get_item` for one spot and
{meth}`~pylabrobot.resources.ItemizedResource.get_items` for a range:

In [ ]:
from pylabrobot.resources import hamilton_96_tiprack_300uL_filter

tip_rack = hamilton_96_tiprack_300uL_filter(name="tips")

a1 = tip_rack.get_item("A1")
first_column = tip_rack.get_items("A1:H1")

assert [spot.get_identifier() for spot in first_column] == [f"{row}1" for row in "ABCDEFGH"]
assert a1.get_identifier() == "A1"
assert a1.has_tip()
tip = a1.get_tip()

The identifiers use transposed spreadsheet notation: rows are letters and columns are numbers.
For example, `A1:H1` selects the first column from top to bottom. A spot's location is relative
to its rack, so moving the rack does not change the spot's local coordinates.

## Setting the initial tip state

Predefined racks are full by default. Pass `with_tips=False` to start with an empty rack, or use
the rack-level methods to describe a partially filled rack:

In [ ]:
tip_rack = hamilton_96_tiprack_300uL_filter(name="tips", with_tips=False)
tip_rack.set_tip_state({"A1": True, "B1": True})

assert tip_rack.get_item("A1").has_tip()
assert not tip_rack.get_item("C1").has_tip()

Use {meth}`~pylabrobot.resources.TipRack.fill` and
{meth}`~pylabrobot.resources.TipRack.empty` to update every spot. See the
{doc}`tip-tracking guide </user_guide/machine-agnostic-features/using-trackers>` for how tip state
is validated and updated during liquid-handling operations.

## Defining tip spots

A custom tip rack definition supplies an ordered grid of `TipSpot` objects. The `make_tip`
callable receives the unique name that the spot assigns to each tip it creates:

In [ ]:
from pylabrobot.resources import Tip, TipRack, TipSpot, create_ordered_items_2d


def make_tip(name: str) -> Tip:
  return Tip(
    name=name,
    has_filter=False,
    total_tip_length=50.0,
    maximal_volume=300.0,
    fitting_depth=8.0,
  )


name = "custom_tip_rack"
tip_rack = TipRack(
  name=name,
  size_x=20.0,
  size_y=20.0,
  size_z=10.0,
  ordered_items=create_ordered_items_2d(
    TipSpot,
    num_items_x=2,
    num_items_y=2,
    dx=1.0,
    dy=1.0,
    dz=0.0,
    item_dx=9.0,
    item_dy=9.0,
    size_x=8.0,
    size_y=8.0,
    make_tip=make_tip,
    name_prefix=name,
  ),
)

assert tip_rack.get_item("A1").get_tip().name == "custom_tip_rack_tipspot_A1#0"
assert tip_rack.get_item("A1").location.x == 1.0
assert tip_rack.get_item("A1").location.y == 10.0
assert tip_rack.get_item("B1").location.y == 1.0

The dimensions and offsets are in millimeters. `dx` sets the x coordinate of the leftmost
column, while `dy` sets the y coordinate of the frontmost row (the bottom row when the rack is
viewed as a grid). Row A is at the back of the rack, where y is largest. `dz` sets the z
coordinate for every spot, and `item_dx` and `item_dy` are the origin-to-origin spacing.

See {class}`~pylabrobot.resources.tip_rack.TipSpot` for the complete API reference.